# MLflow — tracking de experimentos de fine-tuning

## Parámetros de LoRA que vamos a trackear

**`r` — rango de las matrices LoRA**
Controla el tamaño de las matrices adicionales que LoRA añade al modelo.
- Valores bajos (4, 8): aprende menos pero consume poca memoria. Para datasets pequeños.
- Valores altos (32, 64): más capacidad de aprendizaje pero más memoria y riesgo de overfitting.
- El valor más común en producción es r=8 o r=16.

**`lora_alpha` — escala del aprendizaje**
Amplifica o reduce cuánto influyen las matrices LoRA respecto al modelo base.
La regla práctica es siempre lora_alpha = r * 2.
Con r=8 → lora_alpha=16. No hay que pensar mucho en este parámetro.

**`lr` — learning rate (tasa de aprendizaje)**
Velocidad a la que el modelo ajusta los pesos en cada step.
- Demasiado alto (0.01): el modelo no converge, el loss rebota
- Demasiado bajo (0.00001): aprende muy lento, necesitas muchos más epochs
- El valor estándar para LoRA es 2e-4 (0.0002)

**`epochs` — épocas**
Número de veces que el modelo ve todo el dataset completo.
- Con datasets pequeños (como el nuestro de 8 ejemplos) necesitas más epochs
  porque hay pocos datos que ver.
- Con datasets grandes, menos epochs son suficientes.
- Demasiados epochs → overfitting (el modelo memoriza en lugar de aprender)

**`batch_size` — tamaño del lote**
Cuántos ejemplos procesa el modelo a la vez antes de actualizar los pesos.
- Batch size alto: más estable pero más memoria
- Batch size bajo (1): menos memoria, más ruido en el entrenamiento
- Con poca RAM como en nuestro caso, batch_size=1 es lo seguro

## Qué es MLflow

El Git de los experimentos de ML. Cada vez que entrenas con unos parámetros
concretos, MLflow registra qué parámetros usaste y qué métricas obtuviste.

Sin MLflow: pierdes el rastro de qué configuración dio mejores resultados.
Con MLflow: comparas experimentos visualmente y reproduces el mejor.

## Setup

Importamos MLflow y configuramos el experimento.
Un experimento en MLflow agrupa todos los runs relacionados —
en este caso todos los fine-tunings del clasificador de tickets.

In [2]:
import mlflow
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from datasets import Dataset
from sklearn.metrics import f1_score, accuracy_score
from dotenv import load_dotenv
import re

load_dotenv()

# Configuramos el experimento
mlflow.set_experiment("ticket-classifier-finetuning")

print("MLflow configurado")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

MLflow configurado
Tracking URI: sqlite:////Users/alex/Projects/AI_Playground/notebooks/03_finetuning/mlflow.db


## Dataset y evaluación

Mismo dataset de tickets que usamos en el notebook anterior 04_eval/metrics.ipynb.
Añadimos una función de evaluación que devuelve métricas
para que MLflow pueda registrarlas al final de cada run.

In [3]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Dataset de entrenamiento
data = [
    {"input": "Users cannot log in, authentication service returning 500 errors", "output": "severity: P1 | area: backend | escalate: true"},
    {"input": "Button on settings page has wrong color in dark mode", "output": "severity: P4 | area: frontend | escalate: false"},
    {"input": "Database connection pool exhausted, all services affected", "output": "severity: P1 | area: database | escalate: true"},
    {"input": "API gateway returning 429 to enterprise client unexpectedly", "output": "severity: P2 | area: backend | escalate: true"},
    {"input": "Typo in the welcome email subject line", "output": "severity: P4 | area: frontend | escalate: false"},
    {"input": "Payment service down, users cannot complete purchases", "output": "severity: P1 | area: backend | escalate: true"},
    {"input": "Dashboard loads slowly for some users, around 8 seconds", "output": "severity: P2 | area: backend | escalate: false"},
    {"input": "Search results pagination broken on mobile", "output": "severity: P3 | area: frontend | escalate: false"},
]

# Test set para evaluación
test_data = [
    {"ticket": "Database server completely unresponsive", "expected": "P1"},
    {"ticket": "Wrong icon displayed in settings menu", "expected": "P4"},
    {"ticket": "API response time degraded to 3 seconds", "expected": "P2"},
    {"ticket": "Login page crashes on Safari mobile", "expected": "P3"},
    {"ticket": "All microservices down, complete outage", "expected": "P1"},
    {"ticket": "Tooltip text has a typo", "expected": "P4"},
    {"ticket": "Payment processing delayed by 30 seconds", "expected": "P2"},
    {"ticket": "Memory leak causing gradual service degradation", "expected": "P2"},
]

def formatear_ejemplo(ejemplo):
    messages = [
        {"role": "system", "content": "You are a support ticket classifier. Respond only with severity, area and escalation."},
        {"role": "user", "content": ejemplo["input"]},
        {"role": "assistant", "content": ejemplo["output"]}
    ]
    texto = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": texto}

def tokenizar(ejemplo):
    return tokenizer(ejemplo["text"], truncation=True, max_length=256, padding="max_length")

dataset = Dataset.from_list(data)
dataset_formateado = dataset.map(formatear_ejemplo)
dataset_tokenizado = dataset_formateado.map(tokenizar)

print(f"Dataset: {len(dataset_tokenizado)} ejemplos listos")

Map: 100%|██████████| 8/8 [00:00<00:00, 552.49 examples/s]

Dataset: 8 ejemplos listos


## Entrenamiento con MLflow tracking

Ejecutamos tres runs con parámetros diferentes.
MLflow registra automáticamente parámetros y métricas de cada run
para que podamos comparar cuál configuración funciona mejor.

Esto es exactamente lo que falta en el notebook anterior:
sin MLflow no sabemos qué configuración dio el mejor resultado.

In [4]:
def evaluar_modelo(model, tokenizer) -> dict:
    """Evalúa el modelo sobre el test set y devuelve métricas."""
    model.eval()
    model.to("cpu")

    reales = []
    predichas = []

    for ejemplo in test_data:
        messages = [
            {"role": "system", "content": "You are a support ticket classifier. Respond only with severity, area and escalation."},
            {"role": "user", "content": ejemplo["ticket"]}
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=30,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        respuesta = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        match = re.search(r"P[1-4]", respuesta)
        predicha = match.group(0) if match else "UNKNOWN"

        reales.append(ejemplo["expected"])
        predichas.append(predicha)

    accuracy = accuracy_score(reales, predichas)
    f1 = f1_score(reales, predichas, average="weighted", zero_division=0)

    return {"accuracy": accuracy, "f1_weighted": f1}


def entrenar_con_tracking(r: int, lr: float, epochs: int):
    """Entrena el modelo y registra todo en MLflow."""

    with mlflow.start_run(run_name=f"r{r}_lr{lr}_ep{epochs}"):

        # Registrar parámetros
        mlflow.log_param("r", r)
        mlflow.log_param("lora_alpha", r * 2)
        mlflow.log_param("learning_rate", lr)
        mlflow.log_param("epochs", epochs)
        mlflow.log_param("model", MODEL_NAME)

        # Cargar modelo y aplicar LoRA
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
        lora_config = LoraConfig(
            r=r,
            lora_alpha=r * 2,
            target_modules=["q_proj", "v_proj"],
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM"
        )
        model = get_peft_model(model, lora_config)

        # Entrenar
        training_args = TrainingArguments(
            output_dir=f"./runs/r{r}_lr{lr}_ep{epochs}",
            num_train_epochs=epochs,
            per_device_train_batch_size=1,
            learning_rate=lr,
            logging_steps=10,
            fp16=False,
            report_to="none"
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=dataset_tokenizado,
            data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
        )

        trainer.train()

        # Registrar loss final
        final_loss = trainer.state.log_history[-1].get("train_loss", 0)
        mlflow.log_metric("train_loss", final_loss)

        # Evaluar y registrar métricas
        metricas = evaluar_modelo(model, tokenizer)
        mlflow.log_metric("accuracy", metricas["accuracy"])
        mlflow.log_metric("f1_weighted", metricas["f1_weighted"])

        print(f"\nr={r}, lr={lr}, epochs={epochs}")
        print(f"  Loss: {final_loss:.4f} | Accuracy: {metricas['accuracy']:.2f} | F1: {metricas['f1_weighted']:.2f}")

        return metricas


# Tres runs con parámetros diferentes
print("Iniciando experimentos...\n")
entrenar_con_tracking(r=4,  lr=2e-4, epochs=5)
entrenar_con_tracking(r=8,  lr=2e-4, epochs=10)
entrenar_con_tracking(r=16, lr=1e-4, epochs=15)
print("\nExperimentos completados. Abre MLflow UI para comparar.")

Iniciando experimentos...



[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:01<00:00, 219.35it/s]
/Users/alex/Projects/AI_Playground/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,5.139637
20,4.109285
30,3.429411
40,3.111058



r=4, lr=0.0002, epochs=5
  Loss: 3.9473 | Accuracy: 0.00 | F1: 0.00


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 711.27it/s]
/Users/alex/Projects/AI_Playground/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,4.717928
20,3.218415
30,2.289660
40,1.492422
50,0.985961
60,0.807508
70,0.707241
80,0.664658



r=8, lr=0.0002, epochs=10
  Loss: 1.8605 | Accuracy: 0.12 | F1: 0.06


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 759.63it/s]
/Users/alex/Projects/AI_Playground/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,4.768649
20,3.396720
30,2.499737
40,1.781688
50,1.094887
60,0.824946
70,0.689722
80,0.598943
90,0.547631
100,0.480128



r=16, lr=0.0001, epochs=15
  Loss: 1.4703 | Accuracy: 0.62 | F1: 0.62

Experimentos completados. Abre MLflow UI para comparar.
